# Imports and Load Data

In [11]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
import ast
import json
from IPython.display import display

In [2]:
# Load the dataset
df = pd.read_csv("Data\\classified_comment_stance_with_features.csv", encoding='utf-8')

# Display basic info
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())

# Show a small sample
display(df.sample(5))

Shape: (595007, 30)

Column Names: ['comment_id', 'self_text', 'created_time', 'author_name', 'ups', 'downs', 'score', 'post_id', 'subreddit', 'post_self_text', 'post_title', 'post_created_time', 'post_score', 'user_mentions', 'media_links', 'predicted_label', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'sentiment_score', 'factual_score', 'belief_score', 'emotionality_score', 'super_topics', 'super_similarities', 'selected_topics', 'num_selected_topics']


,comment_id,self_text,created_time,author_name,ups,downs,score,post_id,subreddit,post_self_text,...,insult,identity_hate,sentiment_score,factual_score,belief_score,emotionality_score,super_topics,super_similarities,selected_topics,num_selected_topics
559641,k9icuvh,"Huge hideout, a whole 15 guns. Massive stuff r...",2023-11-16 15:14:51,Pacify_,11,0,11,17wj62p,worldnews,NaN,...,0.097098,0.079920,-0.596522,0.560753,0.558708,0.411078,"['Terrorism/HAMAS', 'Arab-Israel Conflict', 'H...","[0.607994, 0.5038374, 0.39155397, 0.39045802, ...","['Terrorism/HAMAS', 'Arab-Israel Conflict']",2
142190,lsp25p8,What did Israel do to warrant those attacks? W...,2024-10-19 15:15:59,kookoomunga24,1,0,1,1g6et4r,IsraelPalestine,Israel supporter here. Many of you have undoub...,...,0.052425,0.084220,-0.847935,0.550084,0.527774,0.576489,"['Arab-Israel Conflict', 'Zionism', 'Terrorism...","[0.5484665, 0.45683613, 0.45060945, 0.4287132,...",['Arab-Israel Conflict'],1
113633,m03ee5u,That is false. The first settlers in Palestine...,2024-12-02 20:51:37,Wonderful-Pilot-2423,1,0,1,1h4mvic,IsraelPalestine,I grew up for 40 years thinking mass antisemit...,...,0.070911,0.246103,-0.534809,0.498826,0.510725,0.819020,"['Zionism', 'Arab-Israel Conflict', 'Settlemen...","[0.6182102, 0.5490745, 0.4468812, 0.4390027, 0...","['Zionism', 'Arab-Israel Conflict']",2
107176,m2bvemj,You obviously want Israel to not exist at all ...,2024-12-16 14:02:25,JaneDi,2,0,2,1hfcbcm,IsraelPalestine,One of the challenges the new government will ...,...,0.011854,0.164082,-0.951308,0.543820,0.582271,0.899714,"['Zionism', 'Arab-Israel Conflict', 'Israel-US...","[0.5570356, 0.5267735, 0.4607838, 0.41464272, ...","['Zionism', 'Arab-Israel Conflict']",2
181226,lllfiz8,The propalis being the ultimate crybullies can...,2024-09-05 07:24:53,TrashSignal04,-1,0,-1,1f96ysx,IsraelPalestine,httpswww.youtube.comwatch?vYNjAjED5RKYt1s The ...,...,0.068504,0.018434,-0.894788,0.523164,0.544985,0.748495,"['Zionism', 'Arab-Israel Conflict', 'Israel-US...","[0.62305254, 0.5214279, 0.5158384, 0.50087553,...","['Zionism', 'Arab-Israel Conflict', 'Israel-US...",4


# Stats

In [5]:
# --- Config ---
affil_col = "predicted_label"
group_a, group_b = "Pro-Israel", "Pro-Palestine"

columns_to_keep = [
    "comment_id","created_time","score","predicted_label","toxic","severe_toxic",
    "obscene","threat","insult","identity_hate","sentiment_score","factual_score",
    "belief_score","emotionality_score","selected_topics"
]

features = [
    "score","toxic","severe_toxic","obscene","threat","insult","identity_hate",
    "sentiment_score","factual_score","belief_score","emotionality_score"
]

df = df[columns_to_keep].dropna(subset=[affil_col])


In [7]:
# --- Helpers ---
def parse_topics(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # Try JSON
    try:
        val = json.loads(s)
        return val if isinstance(val, list) else []
    except Exception:
        pass
    # Try Python-literal list
    try:
        val = ast.literal_eval(s)
        return val if isinstance(val, list) else []
    except Exception:
        return []

def sig_mark(p):
    return "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))


In [12]:
# Standardize affiliation values (optional; comment out if not needed)
df[affil_col] = df[affil_col].astype(str).str.strip()

# Parse topics
df["selected_topics"] = df["selected_topics"].apply(parse_topics)

# Build exploded view for subtopic-level tests
exp = df.explode("selected_topics").rename(columns={"selected_topics":"subtopic"})
exp["subtopic"] = exp["subtopic"].fillna("")

# Include a TOTAL pseudo-topic
exp_total = exp.copy()
exp_total["subtopic"] = "TOTAL"
exp_all = pd.concat([exp, exp_total], ignore_index=True)

# Keep only the two groups
exp_all = exp_all[exp_all[affil_col].isin([group_a, group_b])].copy()

# --- T-tests ---
rows = []
for topic in sorted(exp_all["subtopic"].dropna().unique(), key=lambda x: (x!="TOTAL", x)):
    topic_df = exp_all[exp_all["subtopic"] == topic]

    for feat in features:
        if feat not in topic_df.columns:
            continue

        a = pd.to_numeric(topic_df.loc[topic_df[affil_col]==group_a, feat], errors="coerce").dropna()
        b = pd.to_numeric(topic_df.loc[topic_df[affil_col]==group_b, feat], errors="coerce").dropna()

        # Need at least 2 per group for a stable Welch t-test
        if len(a) < 2 or len(b) < 2:
            pval = np.nan
            tval = np.nan
        else:
            tval, pval = ttest_ind(a, b, equal_var=False, nan_policy="omit")

        rows.append({
            "subtopic": topic,
            "feature": feat,
            f"n_{group_a}": len(a),
            f"n_{group_b}": len(b),
            f"mean_{group_a}": np.nan if len(a)==0 else a.mean(),
            f"mean_{group_b}": np.nan if len(b)==0 else b.mean(),
            "diff_(A-B)": (a.mean() - b.mean()) if (len(a)>0 and len(b)>0) else np.nan,
            "t": tval,
            "p": pval,
            "sig": sig_mark(pval) if pd.notna(pval) else ""
        })

res = pd.DataFrame(rows)

# Sort: significant first (by p), then by topic, then feature
res_sorted = res.sort_values(by=["p","subtopic","feature"], na_position="last").reset_index(drop=True)

# Pretty display: round numeric columns for readability
display_cols = [
    "subtopic","feature",
    f"n_{group_a}", f"n_{group_b}",
    f"mean_{group_a}", f"mean_{group_b}",
    "diff_(A-B)","t","p","sig"
]
pretty = res_sorted[display_cols].copy()
pretty[[f"mean_{group_a}", f"mean_{group_b}", "diff_(A-B)", "t", "p"]] = \
    pretty[[f"mean_{group_a}", f"mean_{group_b}", "diff_(A-B)", "t", "p"]].round(4)

# Notebook-friendly print
with pd.option_context("display.max_rows", 200, "display.max_columns", None, "display.width", 140):
    print(f"Welch t-tests ({group_a} vs {group_b}) by subtopic (including TOTAL).")
    print("* p<.05, ** p<.01, *** p<.001")
    display(pretty)

Welch t-tests (Pro-Israel vs Pro-Palestine) by subtopic (including TOTAL).
* p<.05, ** p<.01, *** p<.001


,subtopic,feature,n_Pro-Israel,n_Pro-Palestine,mean_Pro-Israel,mean_Pro-Palestine,diff_(A-B),t,p,sig
0,Antisemitism,belief_score,29427,30473,0.5399,0.5521,-0.0122,-57.3266,0.0000,***
1,Antisemitism,factual_score,29427,30473,0.5281,0.5447,-0.0166,-73.6138,0.0000,***
2,Antisemitism,identity_hate,29427,30473,0.2336,0.1716,0.0620,56.8608,0.0000,***
3,Antisemitism,toxic,29427,30473,0.6522,0.7184,-0.0661,-57.5547,0.0000,***
4,Arab-Israel Conflict,belief_score,260400,228709,0.5421,0.5511,-0.0089,-132.8665,0.0000,***
5,Arab-Israel Conflict,emotionality_score,260400,228709,0.6876,0.7420,-0.0544,-65.9398,0.0000,***
6,Arab-Israel Conflict,factual_score,260400,228709,0.5305,0.5419,-0.0114,-159.2580,0.0000,***
7,Arab-Israel Conflict,insult,260400,228709,0.0549,0.0488,0.0061,66.1966,0.0000,***
8,Arab-Israel Conflict,obscene,260400,228709,0.0620,0.0538,0.0082,45.8794,0.0000,***
9,Arab-Israel Conflict,severe_toxic,260400,228709,0.0263,0.0213,0.0050,72.7852,0.0000,***


In [19]:
pretty[pretty['feature']=='belief_score']

,subtopic,feature,n_Pro-Israel,n_Pro-Palestine,mean_Pro-Israel,mean_Pro-Palestine,diff_(A-B),t,p,sig
0,Antisemitism,belief_score,29427,30473,0.5399,0.5521,-0.0122,-57.3266,0.0,***
4,Arab-Israel Conflict,belief_score,260400,228709,0.5421,0.5511,-0.0089,-132.8665,0.0,***
13,IDF,belief_score,11132,13283,0.5409,0.5554,-0.0145,-48.4138,0.0,***
15,Israel-US Relations,belief_score,21095,46351,0.5620,0.5697,-0.0078,-42.0852,0.0,***
18,Palestinian Identity,belief_score,8314,9159,0.5359,0.5678,-0.0319,-67.8735,0.0,***
22,TOTAL,belief_score,594641,553341,0.5437,0.5550,-0.0113,-245.8962,0.0,***
31,Terrorism/HAMAS,belief_score,154451,82765,0.5433,0.5554,-0.0120,-113.7137,0.0,***
39,Zionism,belief_score,52617,94506,0.5484,0.5576,-0.0091,-68.8357,0.0,***
43,Civilian Casualties,belief_score,11799,17016,0.5407,0.5511,-0.0104,-37.7575,0.0,***
46,Genocide/War Crimes,belief_score,8436,20994,0.5416,0.5538,-0.0122,-36.4895,0.0,***
